In [5]:
%%writefile tacit_knowledge.json
{
  "id": "tk_dell7920_mem_boot_001",
  "schema_version": "1.3",
  "metadata": {
    "scenario_id": "master_S1_take3",
    "equipment": "Dell Precision 7920",
    "task": "메모리 부팅 트러블슈팅",
    "keywords": ["부팅 실패", "POST", "RDIMM", "ECC RAM", "LED 패턴", "메모리 슬롯"],
    "scenario_title": "조립 후 부팅 실패 진단",
    "source": {
      "video_id": "master_S1_take3.mp4",
      "clip_start": "00:04:12",
      "clip_end": "00:06:48",
      "transcript_ref": "transcripts/master_S1_take3.json"
    }
  },
  "knowledge": {
    "situation": "조립 완료 후 전원은 들어오지만 POST를 통과하지 못하는 상태",
    "situation_source": ["00:04:12"],
    "tacit_insight": "분해 전에 비파괴 점검(케이블 결선 → 전원 LED 점멸 패턴)을 먼저 읽어 원인 범위를 좁힌다.",
    "reasoning": "LED가 깜빡이는 패턴을 보면 메모리 계열 문제로 원인을 좁힐 수 있다.",
    "reasoning_source": ["00:05:02"],
    "reasoning_origin": "utterance",
    "diagnostic_steps": [
      {"order": 1, "action": "전원·패널 커넥터 등 케이블 전체 결선 확인", "evidence": "utterance", "source_utterance": "일단 케이블부터 다 다시 꽂아보고", "timestamp": "00:04:30"},
      {"order": 2, "action": "전원 버튼 LED 점멸 패턴 판독 → 원인 범위 좁히기", "evidence": "utterance", "source_utterance": "여기 불 깜빡이는 거 보면 램 문제예요", "timestamp": "00:05:02"},
      {"order": 3, "action": "RDIMM 탈거 후 접점 세척, 재장착", "evidence": "utterance", "source_utterance": "이거 빼서 접점 닦고 다시 꽂아야", "timestamp": "00:05:48"},
      {"order": 4, "action": "슬롯 재배치", "evidence": "action_only", "source_utterance": null, "timestamp": "00:06:10"},
      {"order": 5, "action": "BIOS 진입하여 메모리 인식 검증", "evidence": "utterance", "source_utterance": "바이오스 들어가서 잡히는지 보고", "timestamp": "00:06:30"}
    ],
    "normal_procedure": "CPU0 → 히트싱크 → ECC RDIMM → HDD 캐리어 → SATA0 → 전원·패널 커넥터 → GPU → POST 확인"
  }
}

Writing tacit_knowledge.json


In [6]:

# -*- coding: utf-8 -*-
"""
RAG 임베딩 모델 성능 비교 하니스
- 입력: tacit_knowledge.json (암묵지 후보 1건 → 의미 단위 청크로 분해)
- 질의: 음성(STT) 스타일의 구어체 한국어 질문 12개 (gold 청크 라벨 포함)
- 지표: Recall@1, Recall@3, MRR, nDCG@3
- 모델: multilingual-e5-large / BAAI-bge-m3 / jhgan-ko-sroberta-multitask (+ BM25 baseline)

사용법:
  python compare_embeddings.py --baseline          # BM25만 (오프라인 검증용)
  python compare_embeddings.py --models all        # 임베딩 모델 3종 (HuggingFace 접근 필요)
"""
import argparse, json, math
from collections import defaultdict

# ---------------------------------------------------------------
# 1. JSON → 검색 청크 (실서비스에서 벡터DB에 들어갈 단위와 동일하게 구성)
# ---------------------------------------------------------------
def build_corpus(path="tacit_knowledge.json"):
    d = json.load(open(path, encoding="utf-8"))
    k, m = d["knowledge"], d["metadata"]
    prefix = f"[{m['equipment']} / {m['task']}] "
    chunks = {}
    chunks["situation"] = prefix + "상황: " + k["situation"]
    chunks["insight"]   = prefix + "암묵지: " + k["tacit_insight"] + " 근거: " + k["reasoning"]
    for s in k["diagnostic_steps"]:
        body = f"진단 {s['order']}단계: {s['action']}"
        if s.get("source_utterance"):
            body += f' (작업자 발화: "{s["source_utterance"]}")'
        chunks[f"step{s['order']}"] = prefix + body
    chunks["procedure"] = prefix + "정상 조립 절차: " + k["normal_procedure"]
    return chunks

# ---------------------------------------------------------------
# 2. STT 스타일 구어체 질의셋 (현장에서 음성으로 들어올 법한 문장)
#    gold = 해당 질의에 반드시 검색되어야 하는 청크 id (복수 가능)
# ---------------------------------------------------------------
QUERIES = [
    ("전원은 들어오는데 부팅이 안 돼요 뭐부터 봐야 하죠",            ["insight", "situation"]),
    ("포스트를 못 넘어가는데 원인 어떻게 좁혀요",                    ["insight", "step2"]),
    ("불이 깜빡깜빡 하는데 이게 무슨 뜻이에요",                      ["step2"]),
    ("LED 점멸 패턴 보고 뭘 알 수 있어요",                           ["step2", "insight"]),
    ("램 문제인 것 같은데 어떻게 해야 돼요",                          ["step3", "step4"]),
    ("메모리 빼서 뭘 해야 하나요",                                   ["step3"]),
    ("접점 닦고 다시 꽂았는데도 안 되면요",                           ["step4", "step5"]),
    ("슬롯 위치 바꿔 끼워봐야 하나요",                               ["step4"]),
    ("케이블은 어떤 순서로 확인해요",                                ["step1"]),
    ("바이오스에서 뭘 확인해야 하죠",                                ["step5"]),
    ("메모리 인식되는지 어디서 봐요",                                ["step5"]),
    ("이 장비 원래 조립 순서가 어떻게 돼요",                          ["procedure"]),
]

# ---------------------------------------------------------------
# 3. 지표
# ---------------------------------------------------------------
def evaluate(rank_fn, corpus, queries, k=3):
    ids = list(corpus.keys())
    per_q, agg = [], defaultdict(float)
    for q, gold in queries:
        ranked = rank_fn(q, corpus)              # [(chunk_id, score) desc]
        top = [cid for cid, _ in ranked]
        r1 = 1.0 if top[0] in gold else 0.0
        rk = 1.0 if any(c in gold for c in top[:k]) else 0.0
        rr = 0.0
        for i, c in enumerate(top):
            if c in gold: rr = 1.0 / (i + 1); break
        dcg  = sum((1.0 if top[i] in gold else 0.0) / math.log2(i + 2) for i in range(min(k, len(top))))
        idcg = sum(1.0 / math.log2(i + 2) for i in range(min(k, len(gold))))
        ndcg = dcg / idcg if idcg else 0.0
        per_q.append((q, gold, top[:k], r1, rk, rr, ndcg))
        agg["R@1"] += r1; agg[f"R@{k}"] += rk; agg["MRR"] += rr; agg[f"nDCG@{k}"] += ndcg
    n = len(queries)
    return {m: v / n for m, v in agg.items()}, per_q

# ---------------------------------------------------------------
# 4-A. BM25 baseline (형태소 분석 기반, 오프라인 실행 가능)
# ---------------------------------------------------------------
def bm25_ranker(corpus):
    from rank_bm25 import BM25Okapi
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    tok = lambda t: [w.form for w in kiwi.tokenize(t)]
    ids = list(corpus.keys())
    bm = BM25Okapi([tok(corpus[i]) for i in ids])
    def rank(q, _):
        sc = bm.get_scores(tok(q))
        return sorted(zip(ids, sc), key=lambda x: -x[1])
    return rank

# ---------------------------------------------------------------
# 4-B. 임베딩 모델 (HuggingFace 접근 가능 환경에서 실행)
# ---------------------------------------------------------------
MODELS = {
    "multilingual-e5-large": dict(name="intfloat/multilingual-e5-large",
                                  q_prefix="query: ", d_prefix="passage: "),
    "bge-m3":                dict(name="BAAI/bge-m3", q_prefix="", d_prefix=""),
    "ko-sroberta-multitask": dict(name="jhgan/ko-sroberta-multitask", q_prefix="", d_prefix=""),
}

def embed_ranker(cfg, corpus):
    from sentence_transformers import SentenceTransformer, util
    model = SentenceTransformer(cfg["name"])
    ids = list(corpus.keys())
    doc_emb = model.encode([cfg["d_prefix"] + corpus[i] for i in ids],
                           normalize_embeddings=True)
    def rank(q, _):
        qe = model.encode([cfg["q_prefix"] + q], normalize_embeddings=True)
        sims = util.cos_sim(qe, doc_emb)[0].tolist()
        return sorted(zip(ids, sims), key=lambda x: -x[1])
    return rank

# ---------------------------------------------------------------
def report(tag, metrics, per_q, verbose=True):
    print(f"\n===== {tag} =====")
    print(" | ".join(f"{m}: {v:.3f}" for m, v in metrics.items()))
    if verbose:
        for q, gold, top, r1, rk, rr, nd in per_q:
            flag = "O" if rk else "X"
            print(f" [{flag}] {q}  gold={gold}  top3={top}")


    # 코랩 셀 직접 실행용
corpus = build_corpus("tacit_knowledge.json")
print(f"청크 {len(corpus)}개 / 질의 {len(QUERIES)}개")

for t in ["bge-m3", "multilingual-e5-large", "ko-sroberta-multitask"]:
    m, pq = evaluate(embed_ranker(MODELS[t], corpus), corpus, QUERIES)
    report(t, m, pq)

청크 8개 / 질의 12개


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


===== ko-sroberta-multitask =====
R@1: 0.667 | R@3: 0.833 | MRR: 0.769 | nDCG@3: 0.729
 [O] 전원은 들어오는데 부팅이 안 돼요 뭐부터 봐야 하죠  gold=['insight', 'situation']  top3=['situation', 'step1', 'step2']
 [O] 포스트를 못 넘어가는데 원인 어떻게 좁혀요  gold=['insight', 'step2']  top3=['insight', 'step2', 'step1']
 [O] 불이 깜빡깜빡 하는데 이게 무슨 뜻이에요  gold=['step2']  top3=['step2', 'insight', 'situation']
 [O] LED 점멸 패턴 보고 뭘 알 수 있어요  gold=['step2', 'insight']  top3=['insight', 'step2', 'step5']
 [X] 램 문제인 것 같은데 어떻게 해야 돼요  gold=['step3', 'step4']  top3=['step2', 'insight', 'situation']
 [O] 메모리 빼서 뭘 해야 하나요  gold=['step3']  top3=['insight', 'situation', 'step3']
 [X] 접점 닦고 다시 꽂았는데도 안 되면요  gold=['step4', 'step5']  top3=['step3', 'step1', 'procedure']
 [O] 슬롯 위치 바꿔 끼워봐야 하나요  gold=['step4']  top3=['step3', 'step4', 'insight']
 [O] 케이블은 어떤 순서로 확인해요  gold=['step1']  top3=['step1', 'insight', 'procedure']
 [O] 바이오스에서 뭘 확인해야 하죠  gold=['step5']  top3=['step5', 'insight', 'step3']
 [O] 메모리 인식되는지 어디서 봐요  gold=['step5']  top3=['step5', 'in